In [37]:
import os 
import certifi 
from dotenv import load_dotenv


from langchain_groq import ChatGroq
from langsmith import Client
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain.tools import tool
import requests

In [38]:
from langchain_classic.agents import create_react_agent, AgentExecutor

In [39]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROK_API_KEY = os.getenv("GROK_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [40]:
search_tool = TavilySearchResults(max_results=2)

In [41]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [ ]:
#testing the search tool
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'AI News | Latest News | Insights Powering AI-Driven Business Growth',
  'url': 'https://www.artificialintelligence-news.com',
  'content': 'Governance, Regulation & Policy\n\nAugust 4, 2026\n\n### Google’s Gemini 3.6 Flash targets enterprise agent token costs\n\nAI in Action\n\nJuly 21, 2026\n\n### AWS GraphRAG deployment cuts drug research cycles by 87%\n\nHealthcare & Wellness AI\n\nJuly 9, 2026\n\n#### Industries\n\n### Zuckerberg details Meta’s personal AI superintelligence strategy\n\nAI and Us\n\nJuly 30, 2026\n\n### Google’s Gemini 3.6 Flash targets enterprise agent token costs\n\nAI in Action\n\nJuly 21, 2026\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention [...] 19 October 2026 

In [43]:
# ==========================================
# LLM
# ==========================================

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROK_API_KEY
)

In [ ]:
#testing the llm
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI program go on a diet?\n\nBecause it wanted to lose some bytes! (get it?)', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 41, 'total_tokens': 64, 'completion_time': 0.060390768, 'completion_tokens_details': None, 'prompt_time': 0.001970745, 'prompt_tokens_details': None, 'queue_time': 0.056280504, 'total_time': 0.062361513}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd71f-5700-7100-8dd8-1334df415e03-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 23, 'total_tokens': 64})

In [ ]:
# ==========================================
# PROMPT
# ==========================================



client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True,
)

In [46]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [47]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [48]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [49]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [50]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})




> Entering new AgentExecutor chain...
Thought: To find the capital of India and its current weather, I first need to identify the capital of India. I can use the tavily_search_results_json tool to search for this information.

Action: tavily_search_results_json
Action Input: "capital of India"[{'title': 'What is the capital of India? States and union territories explained.', 'url': 'https://www.usatoday.com/story/news/world/2023/05/24/what-is-the-capital-of-india/70195720007', 'content': "## What is the capital of India?\n\nThe capital of India is New Delhi, located in the north-central part of the country to the west of the Yamuna River.\n\nCalcutta (now Kolkata, the capital of West Bengal) was the country’s capital until 1911 when King George V declared Delhi the new capital and construction of New Delhi began.\n\nWhen the national government achieved independence in 1947, New Delhi became the capital.\n\nMumbai, the state capital of Maharashtra, is often considered the financial c

In [51]:
print(response["output"])

The capital of India is New Delhi, and its current weather is 30°C with patchy rain nearby and a humidity of 68%.
